In [1]:
import pandas as pd
import os
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk

try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
    nltk.download('wordnet')
    nltk.download('punkt')

RAW_DATA_PATH = '/kaggle/input/datasets/entenam/reddit-mental-health-dataset/Original Reddit Data/raw data'
OUTPUT_FOLDER = '/kaggle/working/'
OUTPUT_FILENAME = os.path.join(OUTPUT_FOLDER, 'cleaned_labelled_raw_data.csv')


def clean_text(text):
    if not isinstance(text, str):
        return ""
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2]
    return " ".join(cleaned_tokens)


def main():
    print("Starting data processing...")
    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)

    all_data = []
    label_mapping = {
        'ani': 'anxiety', 'dep': 'depression', 'lon': 'lonely',
        'mhj': 'mentalhealth', 'swj': 'suicidewatch'
    }

    for root, _, files in os.walk(RAW_DATA_PATH):
        for filename in files:
            if filename.endswith('.csv'):
                file_path = os.path.join(root, filename)
                prefix = filename[:3].lower()
                label = label_mapping.get(prefix, 'unknown')
                print(f"Processing {file_path} with label '{label}'...")

                try:
                    df = pd.read_csv(file_path)
                    
                    if 'selftext' in df.columns:
                        
                        for text in df['selftext']:
                            all_data.append({'text': text, 'label': label})
                    else:
                        print(f"  [Warning] Column 'selftext' not found in {filename}. Skipping.")

                except Exception as e:
                    print(f"  [Error] Could not process {filename}. Reason: {e}")

    print("\nCombining all data into a single DataFrame...")
    master_df = pd.DataFrame(all_data)
    master_df.dropna(subset=['text'], inplace=True)
    master_df.reset_index(drop=True, inplace=True)
    print(f"Total posts collected: {len(master_df)}")

    print("Cleaning and preprocessing text... (This may take a while)")
    master_df['cleaned_text'] = master_df['text'].apply(clean_text)

    print(f"Saving cleaned data to {OUTPUT_FILENAME}...")
    master_df.to_csv(OUTPUT_FILENAME, index=False)
    
    print("\nProcessing complete!")
    print(f"Your cleaned file is ready at '{OUTPUT_FILENAME}'")

main()

Starting data processing...
Processing /kaggle/input/datasets/entenam/reddit-mental-health-dataset/Original Reddit Data/raw data/2020/Nov 20/swnov20.csv with label 'unknown'...
Processing /kaggle/input/datasets/entenam/reddit-mental-health-dataset/Original Reddit Data/raw data/2020/Nov 20/depnov20.csv with label 'depression'...
Processing /kaggle/input/datasets/entenam/reddit-mental-health-dataset/Original Reddit Data/raw data/2020/Nov 20/mhov20.csv with label 'unknown'...
Processing /kaggle/input/datasets/entenam/reddit-mental-health-dataset/Original Reddit Data/raw data/2020/Nov 20/anxinov20.csv with label 'unknown'...
Processing /kaggle/input/datasets/entenam/reddit-mental-health-dataset/Original Reddit Data/raw data/2020/Nov 20/lonenov20.csv with label 'lonely'...
Processing /kaggle/input/datasets/entenam/reddit-mental-health-dataset/Original Reddit Data/raw data/2020/Aug 20/Swaug20 copy.csv with label 'unknown'...
Processing /kaggle/input/datasets/entenam/reddit-mental-health-data

In [2]:
import pandas as pd

file_path = '/kaggle/working/cleaned_labelled_raw_data.csv'

print(f"Loading data from {file_path}...")
df = pd.read_csv(file_path)
print(f"Original number of posts: {len(df)}")

print("\nFiltering out rows with the 'unknown' label...")
df_clean = df[df['label'] != 'unknown'].copy()
print(f"Number of posts after filtering: {len(df_clean)}")

print(f"\nSaving the cleaned data back to {file_path}...")
df_clean.to_csv(file_path, index=False)

print("\nFile has been successfully updated.")

Loading data from /kaggle/working/cleaned_labelled_raw_data.csv...
Original number of posts: 1797016

Filtering out rows with the 'unknown' label...
Number of posts after filtering: 1024932

Saving the cleaned data back to /kaggle/working/cleaned_labelled_raw_data.csv...

File has been successfully updated.
